# Time Series Practice 1

Validated analysis notebook for AirPassengers. Exercise 6 is a separate Word report owned by TuVien.

## 0. Reproducibility and configuration

In [1]:
import json
import platform
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import statsmodels

from practice_1.analysis.acf_pacf import compute_acf_table, compute_pacf_table
from practice_1.data.air_passengers import calculate_sha256, load_air_passengers
from practice_1.visualization.diagnostic_plots import plot_acf_diagnostic, plot_pacf_diagnostic, plot_time_series

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
config = json.loads((project_root / 'config' / 'practice_1.json').read_text(encoding='utf-8'))
raw_path = project_root / 'datasets' / 'raw' / config['dataset']['raw_filename']
figures_path = project_root / 'outputs' / 'figures'
results_path = project_root / 'outputs' / 'results'
figures_path.mkdir(parents=True, exist_ok=True)
results_path.mkdir(parents=True, exist_ok=True)
print({'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'matplotlib': matplotlib.__version__, 'statsmodels': statsmodels.__version__})

{'python': '3.12.15', 'numpy': '2.5.3', 'pandas': '2.3.3', 'matplotlib': '3.11.2', 'statsmodels': '0.14.6'}


## 1. Dataset provenance, loading and validation

In [2]:
actual_sha256 = calculate_sha256(raw_path)
assert actual_sha256 == config['dataset']['sha256']
air_passengers = load_air_passengers(raw_path)
print('SHA-256:', actual_sha256)
print('Shape:', air_passengers.shape)
print('Date range:', air_passengers.index.min(), 'to', air_passengers.index.max())
print('Missing values:', int(air_passengers.isna().sum().sum()))
print('Duplicate timestamps:', int(air_passengers.index.duplicated().sum()))
display(air_passengers.head())
air_passengers.info()

SHA-256: bdb98adbd418a6de6842a742e0602f363c3b841c26677e7e61a1e055e9509bd8
Shape: (144, 1)
Date range: 1949-01-01 00:00:00 to 1960-12-01 00:00:00
Missing values: 0
Duplicate timestamps: 0


,passengers_thousands
date,
1949-01-01,112.0
1949-02-01,118.0
1949-03-01,132.0
1949-04-01,129.0
1949-05-01,121.0


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 144 entries, 1949-01-01 to 1960-12-01
Freq: MS
Data columns (total 1 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   passengers_thousands  144 non-null    float64
dtypes: float64(1)
memory usage: 6.3 KB


## 2. Exercise 1 — ACF/PACF

In [3]:
series = air_passengers['passengers_thousands']
max_lag = config['correlation']['max_lag']
alpha = config['correlation']['alpha']
pacf_method = config['correlation']['pacf_method']
plot_time_series(air_passengers, figures_path / '01_time_series_overview.png')
acf_table = compute_acf_table(series, max_lag=max_lag, alpha=alpha)
pacf_table = compute_pacf_table(series, max_lag=max_lag, alpha=alpha, method=pacf_method)
acf_table.to_csv(results_path / 'acf_values.csv', index=False)
pacf_table.to_csv(results_path / 'pacf_values.csv', index=False)
plot_acf_diagnostic(series, figures_path / '02_acf_40_lags.png', max_lag=max_lag, alpha=alpha)
plot_pacf_diagnostic(series, figures_path / '03_pacf_40_lags.png', max_lag=max_lag, alpha=alpha, method=pacf_method)
display(acf_table.head(13))
display(pacf_table.head(13))

,lag,acf,lower_ci,upper_ci
0,0,1.000000,1.000000,1.000000
1,1,0.948047,0.784717,1.111378
2,2,0.875575,0.602389,1.148761
3,3,0.806681,0.466779,1.146583
4,4,0.752625,0.365002,1.140249
5,5,0.713770,0.288948,1.138592
6,6,0.681734,0.226041,1.137427
7,7,0.662904,0.180771,1.145038
8,8,0.655610,0.149746,1.161475
9,9,0.670948,0.142904,1.198993


,lag,pacf,lower_ci,upper_ci
0,0,1.000000,1.000000,1.000000
1,1,0.948047,0.784717,1.111378
2,2,-0.229422,-0.392752,-0.066092
3,3,0.038148,-0.125183,0.201478
4,4,0.093785,-0.069545,0.257116
5,5,0.073607,-0.089724,0.236937
6,6,0.007728,-0.155603,0.171058
7,7,0.125597,-0.037733,0.288927
8,8,0.089951,-0.073379,0.253282
9,9,0.232489,0.069158,0.395819


In [4]:
acf_nonzero = acf_table.query('lag > 0').copy()
pacf_nonzero = pacf_table.query('lag > 0').copy()
acf_significant = acf_nonzero.query('lower_ci > 0 or upper_ci < 0')['lag'].astype(int).tolist()
pacf_significant = pacf_nonzero.query('lower_ci > 0 or upper_ci < 0')['lag'].astype(int).tolist()
seasonal_lags = acf_table[acf_table['lag'].isin([12, 24, 36])][['lag', 'acf', 'lower_ci', 'upper_ci']]
top_pacf = pacf_nonzero.assign(abs_pacf=pacf_nonzero['pacf'].abs()).nlargest(10, 'abs_pacf')[['lag', 'pacf']]
findings = {
    'acf_lag_1': float(acf_table.loc[acf_table['lag'] == 1, 'acf'].iloc[0]),
    'acf_significant_lags': acf_significant,
    'pacf_significant_lags': pacf_significant,
    'seasonal_acf': {str(int(row.lag)): float(row.acf) for row in seasonal_lags.itertuples()},
    'top_absolute_pacf_lags': [int(value) for value in top_pacf['lag'].tolist()],
}
(results_path / 'exercise_01_findings.json').write_text(json.dumps(findings, indent=2), encoding='utf-8')
display(seasonal_lags)
display(top_pacf)
print(json.dumps(findings, indent=2))

,lag,acf,lower_ci,upper_ci
12,12,0.760395,0.161505,1.359285
24,24,0.532190,-0.213970,1.278350
36,36,0.337024,-0.463886,1.137933


,lag,pacf
1,1,0.948047
13,13,-0.539691
9,9,0.232489
2,2,-0.229422
11,11,0.171274
10,10,0.166051
25,25,-0.162746
12,12,-0.135431
7,7,0.125597
22,22,-0.100179


{
  "acf_lag_1": 0.9480473407524913,
  "acf_significant_lags": [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14
  ],
  "pacf_significant_lags": [
    1,
    2,
    9,
    10,
    11,
    13
  ],
  "seasonal_acf": {
    "12": 0.7603950422625557,
    "24": 0.532189830465802,
    "36": 0.33702359902053575
  },
  "top_absolute_pacf_lags": [
    1,
    13,
    9,
    2,
    11,
    10,
    25,
    12,
    7,
    22
  ]
}


### Exercise 1 interpretation

- The validated series contains 144 monthly observations from January 1949 through December 1960, with no missing values or duplicate timestamps.
- The overview plot shows a clear upward trend, a repeating annual pattern and seasonal fluctuations whose amplitude increases with the series level. The raw series is therefore not visually stationary.
- Lag 1 has an ACF of 0.9480. The ACF remains strongly positive and decays slowly, so AirPassengers has strong temporal dependence and does not resemble white noise.
- Local ACF peaks recur at lag 12 (0.7604), lag 24 (0.5322) and lag 36 (0.3370). This repeated spacing supports a 12-month seasonal structure, although the widening Bartlett interval means the later peaks should not be treated as separate significance claims without additional analysis.
- PACF lag 1 is the strongest positive direct association. PACF lag 13 is the largest later absolute spike and is negative; significant PACF spikes also occur at lags 2, 9, 10 and 11 under the configured interval.
- Lag 1 is the most important short-term lag. Lag 12 is the most important seasonal reference because of the recurring ACF peaks at its multiples.
- The slow ACF decay can reflect trend and seasonality in the raw non-stationary series. These plots are diagnostic evidence and are not used here to select an ARIMA order.